# 🎬 Keep One Person
Remove people from a video as if they had never been there - **keep just one person** (everyone else goes), or **remove the people you choose** (everyone else stays). The sound stays as it was.

1. In **Google Drive**, put your **video** (and, for *keep one person*, a few **photos of that person**) in the folder **Keep One Person** (it is created for you the first time).
2. Press the **▶ play button** below. Click **Run anyway** if asked, then **allow Google Drive**.
3. After about 5 minutes the app appears below. Press **Start with these files** and follow it.

Finished videos appear in **Google Drive › Keep One Person › Finished**. This notebook uses an **A100 GPU** (Colab Pro); if none is free, choose another under *Runtime › Change runtime type*.

In [ ]:
#@title ▶ Press this play button, then wait about 5 minutes
#@markdown The app opens below this box (and as a link ending in gradio.live). Keep this tab open while it works.
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
import os, subprocess, sys, importlib

def step(title, cmd, required=True):
    print("… " + title, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout or "")[-2000:], (r.stderr or "")[-2000:])
        if required:
            raise RuntimeError("Setup step failed: " + title + ". Please send a screenshot of this box.")
        print("⚠️ " + title + " failed - continuing without it.")
        return False
    print("✅ " + title, flush=True)
    return True

# 1) GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is connected. Menu: Runtime > Change runtime type > A100 GPU "
                       "(or L4 / T4) > Save, then press the play button again.")
print("✅ GPU: " + torch.cuda.get_device_name(0))

# 2) Google Drive (saves progress of long videos)
if SAVE_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs("/content/drive/MyDrive/Keep One Person", exist_ok=True)
        print("✅ Google Drive connected. Your video and photos go in the Drive folder "
              "'Keep One Person'; finished videos appear in 'Keep One Person/Finished'.")
    except Exception as e:
        print("⚠️ Google Drive not connected - progress will not survive a disconnect:", e)

# 3) the program - always the latest version
CODE = "/content/keep_one_person"
if os.path.isdir(CODE + "/.git"):
    step("Updating the program", f"git -C {CODE} fetch -q --depth 1 origin keep-one-person && git -C {CODE} reset -q --hard FETCH_HEAD")
else:
    step("Downloading the program", f"git clone -q --depth 1 -b keep-one-person https://github.com/grok69420/minimal-launcher.git {CODE}")

# 4) AI engines (once per session)
os.makedirs("/content/repos", exist_ok=True)
if not os.path.isdir("/content/repos/sam2/sam2"):
    step("Downloading the tracking engine", "rm -rf /content/repos/sam2 && git clone -q https://github.com/facebookresearch/sam2.git /content/repos/sam2")
if not os.path.isdir("/content/repos/ProPainter/model"):
    step("Downloading the background engine", "rm -rf /content/repos/ProPainter && git clone -q https://github.com/sczhou/ProPainter.git /content/repos/ProPainter")
if not os.path.exists("/content/.kop_packages_v5"):
    step("Installing packages (2-3 min)", "pip -q install hydra-core iopath av addict einops future timm yapf "
         "imageio-ffmpeg onnxruntime ultralytics==8.4.194 gradio==6.31.0 diffusers==0.33.1")
    step("Installing face recognition", "pip -q install insightface==2.1", required=False)
    open("/content/.kop_packages_v5", "w").write("ok")
if not os.path.isdir("/content/repos/MiniMax-Remover"):
    step("Downloading the realistic-fill engine", "git clone -q --depth 1 https://github.com/zibojia/MiniMax-Remover.git /content/repos/MiniMax-Remover", required=False)
if not os.path.exists("/content/repos/minimax_weights/transformer"):
    print("… Downloading the realistic-fill model (about 2.5 GB)", flush=True)
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id="zibojia/minimax-remover", local_dir="/content/repos/minimax_weights",
                          allow_patterns=["vae/*", "transformer/*", "scheduler/*"])
        print("✅ Realistic-fill model ready")
    except Exception as e:
        print("⚠️ Realistic-fill model not available - the fast fill will be used:", e)
for f in ["ProPainter.pth", "raft-things.pth", "recurrent_flow_completion.pth"]:
    if not os.path.exists("/content/repos/ProPainter/weights/" + f):
        step("Downloading " + f, "wget -q -nc -P /content/repos/ProPainter/weights "
             "https://github.com/sczhou/ProPainter/releases/download/v0.1.0/" + f)
if not os.path.exists("/content/repos/sam2/checkpoints/sam2.1_hiera_large.pt"):
    step("Downloading the main AI model (900 MB)", "wget -q -nc -P /content/repos/sam2/checkpoints "
         "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt")

# 5) start the app
os.chdir("/content")
if CODE not in sys.path:
    sys.path.insert(0, CODE)
import remove_people, app
importlib.reload(remove_people)
importlib.reload(app)
print("✅ Starting the app. Use it below, or open the link ending in gradio.live in a new tab.")
app.demo.queue().launch(share=True, debug=True, inline=True, height=1000, css=app.CSS,
                        allowed_paths=[app.CACHE, "/content/drive"])